In [174]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


In [58]:
def extract_series(df):
    df_temp = df
    # Convert 'id' column to string type
    df_temp['id'] = df_temp['id'].astype(str)
    # Add a new column 'series' which is the first three digits of 'id'
    df_temp['series'] = df_temp['id'].str[:3].astype(int)

    # Read the series.csv file
    series = pd.read_csv('Series Name.csv')

    # Merge data with series on the 'series' column from data and 'series_id' column from series
    df_temp_1 = df_temp.merge(series, left_on='series', right_on='Series Name ID', how='left')

    cols = df_temp_1.columns.tolist()
    cols.insert(1, cols.pop(cols.index('Series Name')))
    df_temp_1 = df_temp_1[cols]
    df_temp_1.drop(columns=['Series Name ID', 'series'], inplace=True)

    return df_temp_1

In [59]:
def extract_contry(df):
    df_temp = df
    # Add a new column 'country' which is the fourth and fifth digits of 'id'
    df_temp['country'] = df_temp['id'].str[3:5].astype(int)

# Read the country.csv file
    country = pd.read_csv('Country Name.csv')

# Merge data_series with country on the 'country' column from data_series and 'country_id' column from country
    df_temp_1 = df_temp.merge(country, left_on='country', right_on='Country Name ID', how='left')

# Insert 'location' column into the correct position
    cols = df_temp_1.columns.tolist()
    cols.insert(2, cols.pop(cols.index('Country Name')))
    df_temp_1 = df_temp_1[cols]

# Drop unnecessary columns
    df_temp_1.drop(columns=['Country Name ID', 'country'], inplace=True)
    
    return df_temp_1

In [60]:
def extract_category(df):
    df_temp = df
    # Add a new column 'country' which is the fourth and fifth digits of 'id'
    df_temp['category'] = df_temp['id'].str[5:7].astype(int)

# Read the country.csv file
    category = pd.read_excel('category_id.xlsx')

# Rename the 'id' column in the country dataframe to 'country_id'
    category.rename(columns={'id': 'category_id'}, inplace=True)

# Merge data_series with country on the 'country' column from data_series and 'country_id' column from country
    df_temp_1 = df_temp.merge(category, left_on='category', right_on='category_id', how='left')

# Insert 'location' column into the correct position
    cols = df_temp_1.columns.tolist()
    cols.insert(3, cols.pop(cols.index('Category')))
    df_temp_1 = df_temp_1[cols]

# Drop unnecessary columns
    df_temp_1.drop(columns=['category_id', 'category'], inplace=True)

    return df_temp_1

In [61]:
prosperity = pd.read_csv('prosperity.csv')
prosperity.columns = ['year', 'prosperity']

In [62]:
data = pd.read_csv('cleaned_data.csv')
data['id'] = data['id'].astype(str)
data['id'] = data['id'].apply(lambda x: f'{int(x):013}' if pd.notnull(x) else x)
#nf_data = data[data['id'].str[5:7] != '05']
data = data.set_index('id')

In [63]:
# Create a dataframe to store the id and p-values
pvalues_df = pd.DataFrame(columns=['id', 'pvalue'])

for col in range(len(data)):

    # Ensure y and x have the same length
    x = data.iloc[col].values.reshape(-1, 1)
    y = prosperity['prosperity']

    # Add a constant to the model (intercept)
    x = sm.add_constant(x)
    model = sm.OLS(y, x).fit()

    pvalues_df.loc[col, 'id'] = data.index[col]
    
    # Check if model.pvalues has an index 1
    if len(model.pvalues) > 1:
        pvalues_df.loc[col, 'pvalue'] = model.pvalues[1]  # Extract the p-value for the predictor variable
    else:
        pvalues_df.loc[col, 'pvalue'] = 1 # Assign NaN if p-value is not available



/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_857/481056520.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pvalues_df.loc[col, 'pvalue'] = model.pvalues[1]  # Extract the p-value for the predictor variable
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_857/481056520.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pvalues_df.loc[col, 'pvalue'] = model.pvalues[1]  # Extract the p-value for the predictor variable
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_857/481056520.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future versio

In [64]:
pvalues_threshold = 0.05 / len(data)
data_significant = pvalues_df[pvalues_df['pvalue'] < pvalues_threshold]

In [ ]:
# Convert 'pvalue' column to numeric
pvalues_df['pvalue'] = pd.to_numeric(pvalues_df['pvalue'], errors='coerce')


0        0050104000005
3        0100104000010
4        0130101000013
5        0140101000014
7        0160104000016
             ...      
10207    5413201025111
10208    5613202025131
10209    6093204025179
10210    6213201025191
10211    6293203025199
Name: id, Length: 7420, dtype: object

In [ ]:
temp = data.reset_index()

# Get the top 10 rows with the smallest p-values
#nf_data_top10_pvalues = pvalues_df.nsmallest(100, 'pvalue')
#nf_data_top10_pvalues = nf_data_significant

data_significant_db = temp[temp['id'].isin(data_significant['id'])]

,id,1993,1994,1995,1996,1997,1998,1999,2000,2001,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,0050104000005,1.878951e+07,1.889035e+07,1.895606e+07,1.878242e+07,1.821171e+07,1.780392e+07,1.437832e+07,1.410682e+07,1.416699e+07,...,1.643133e+08,2.068983e+08,1.903312e+08,1.788537e+08,2.378150e+08,2.479358e+08,2.323007e+08,1.822300e+08,2.554767e+08,2.653056e+08
1,0080104000008,2.889968e+06,2.514651e+06,2.155011e+06,2.452002e+06,1.920184e+06,1.070375e+06,1.455843e+06,2.065691e+06,2.013539e+06,...,1.986275e+07,1.129754e+07,1.389760e+07,2.317650e+07,3.041629e+07,1.935435e+07,1.161018e+06,2.714217e+06,1.977231e+07,2.049601e+07
2,0090104000009,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,4.721533e+05,3.051536e+05,3.888372e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.373352e+05,1.429700e+05
3,0100104000010,2.147732e+07,2.290039e+07,2.382507e+07,2.183181e+07,2.402106e+07,2.278064e+07,2.805942e+07,2.431737e+07,2.400059e+07,...,4.450926e+07,4.696891e+07,5.182773e+07,4.399937e+07,4.422070e+07,5.172933e+07,4.838257e+07,4.737678e+07,5.125604e+07,5.241063e+07
4,0130101000013,1.494610e+02,1.568350e+02,1.583150e+02,1.576030e+02,1.587610e+02,1.559420e+02,1.563650e+02,1.543100e+02,1.524710e+02,...,8.406900e+01,8.104300e+01,7.813000e+01,7.530000e+01,7.302100e+01,7.096700e+01,6.887700e+01,6.659900e+01,6.533900e+01,5.661966e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10208,5613202025131,1.037000e+00,1.037000e+00,1.037000e+00,1.037000e+00,1.036000e+00,1.036000e+00,1.036000e+00,1.036000e+00,1.036000e+00,...,1.035000e+00,1.035000e+00,1.035000e+00,1.035000e+00,1.035000e+00,1.035000e+00,1.035000e+00,1.035000e+00,1.035000e+00,1.034421e+00
10209,6093204025179,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,3.702000e+04,3.702000e+04,3.702400e+04,3.902700e+04,4.103000e+04,4.303500e+04,4.104000e+04,4.404500e+04,4.704700e+04,4.425999e+04
10210,6213201025191,7.744250e+05,7.986610e+05,8.294970e+05,8.569470e+05,8.882380e+05,9.132050e+05,9.451620e+05,9.955650e+05,1.049212e+06,...,2.092645e+06,2.093997e+06,2.067683e+06,1.984736e+06,1.985622e+06,2.074149e+06,2.160983e+06,2.229006e+06,2.297475e+06,2.433901e+06
10211,6293203025199,4.125000e+01,4.125000e+01,4.125000e+01,4.125000e+01,4.375000e+01,4.375000e+01,4.375000e+01,4.375000e+01,4.375000e+01,...,4.937500e+01,4.937500e+01,4.937500e+01,4.937500e+01,6.750000e+01,6.750000e+01,6.750000e+01,6.750000e+01,6.750000e+01,6.750000e+01


In [75]:
data_significant_seires = extract_series(data_significant_db)
data_significant_country = extract_contry(data_significant_seires)
data_significant_category = extract_category(data_significant_country)

/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_857/571456676.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['id'] = df_temp['id'].astype(str)
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_857/571456676.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['series'] = df_temp['id'].str[:3].astype(int)


In [ ]:
data_significant_category.to_excel('top10.xlsx')

In [78]:
data_significant_category.groupby('Series Name').count()['id'].sort_values(ascending=False).to_csv('significant.csv')

In [84]:
data_significant_category.groupby('Country Name').count()['id'].sort_values(ascending=False)

Country Name
Bangladesh          241
Nepal               237
India               236
Korea, Rep.         236
China               230
Kenya               228
Pakistan            226
Tanzania            225
Ethiopia            222
Uganda              217
Germany             212
Singapore           207
Sweden              201
Australia           201
France              200
Madagascar          197
Italy               194
Norway              194
United States       193
Netherlands         192
Denmark             190
Congo, Dem. Rep.    188
Switzerland         188
United Kingdom      187
Finland             184
Canada              183
Chad                182
Malawi              180
Mozambique          178
Japan               171
Austria             170
Nigeria             170
Myanmar             157
Belgium             156
New Zealand         152
Ireland             149
Zimbabwe            108
Afghanistan         104
Yemen, Rep.          85
South Sudan          49
Name: id, dtype: int64

In [153]:
data_significant_db_transposed = data_significant_db.transpose().reset_index()
data_significant_db_transposed.columns = data_significant_db_transposed.iloc[0]
data_significant_clean = data_significant_db_transposed.drop(0)

data_significant_clean.set_index('id', inplace=True)
data_significant_clean.index.name = None


In [163]:
data_significant_clean = data_significant_clean.apply(pd.to_numeric, errors='coerce')

In [171]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Fit and transform the data
data_significant_clean_standardized = pd.DataFrame(scaler.fit_transform(data_significant_clean), 
                                                   columns=data_significant_clean.columns, 
                                                   index=data_significant_clean.index)

In [ ]:
# Define X and y
X = data_significant_clean_standardized
y = prosperity['prosperity']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Lasso model
lasso = Lasso(alpha=0.1)

# Fit the model
lasso.fit(X_train, y_train)

# Predict on the test set
y_pred = lasso.predict(X_test)

Coefficients: [ 0.  0. -0. ...  0.  0.  0.]


In [181]:
# Get the coefficients from the Lasso model
coefficients = lasso.coef_

# Create a DataFrame to display the coefficients with their corresponding column names
coefficients_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': coefficients})

# Print the DataFrame
coefficients_df.sort_values(by = 'Coefficient', ascending=False).to_csv('lasso.csv')

In [185]:
lasso_data = coefficients_df[abs(coefficients_df['Coefficient']) > 0]

temp = data.reset_index()
lasso_db = temp[temp['id'].isin(lasso_data['Feature'])]
lasso_seires = extract_series(lasso_db)
lasso_country = extract_contry(lasso_seires)
lasso_category = extract_category(lasso_country)

/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_857/571456676.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['id'] = df_temp['id'].astype(str)
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_857/571456676.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['series'] = df_temp['id'].str[:3].astype(int)


In [186]:
lasso_category.to_excel('lasso_data.xlsx')